# 🚀 Análisis de Logs de Servidores NASA con PySpark

Notebook de **exploración interactiva**. La vía **reproducible** del proyecto es el script
`src/nasa_logs_pipeline.py` (mismo pipeline, ejecutable sin Jupyter y con pruebas).

Este notebook reutiliza el regex y la configuración de entorno de `src/`, para que no existan
dos versiones de la lógica que puedan divergir.

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del proyecto (funciona si abres el notebook desde la carpeta NasaHTTP/).
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = Path(r"C:\Users\junio\Documents\Python Ejecutables\Potafolio DSc\NasaHTTP")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Fija JAVA_HOME, SPARK_HOME y HADOOP_HOME ANTES de crear la SparkSession.
# Los tres son causa habitual de fallo de Spark en Windows; ver el README.
from nasa_logs_pipeline import (
    build_spark,
    configure_hadoop_home,
    configure_java_home,
    configure_spark_home,
)

JAVA_HOME = configure_java_home()
SPARK_HOME = configure_spark_home()
HADOOP_HOME = configure_hadoop_home()

print("JAVA_HOME   =", JAVA_HOME)
print("SPARK_HOME  =", SPARK_HOME)
print("HADOOP_HOME =", HADOOP_HOME)

In [ ]:
from pyspark.sql import functions as F

from nasa_logs import (
    GROUP_ENDPOINT,
    GROUP_IP,
    GROUP_METHOD,
    GROUP_PROTOCOL,
    GROUP_SIZE,
    GROUP_STATUS,
    GROUP_TIMESTAMP,
    LOG_PATTERN,
)

# ANSI desactivado: el log trae tamaños "-" que no castean a int.
spark = build_spark("Nasa_Log_Analysis")
spark.sparkContext.setLogLevel("ERROR")
print("✅ Sesión de Spark iniciada")

In [ ]:
from nasa_logs import resolve_log_path

# Resolución en cascada: --log > NASA_LOG_PATH > data/access.log
log_path = resolve_log_path()
print(f"Dataset: {log_path} ({log_path.stat().st_size / 1e6:.1f} MB)")

raw_logs = spark.read.text(str(log_path))
print(f"Líneas leídas: {raw_logs.count():,}")

In [ ]:
# Regex robusto (definido una sola vez en src/nasa_logs.py):
#   - el protocolo HTTP es OPCIONAL  -> no se pierden peticiones tipo "GET /ruta"
#   - se usa \s+ en vez de un espacio -> soporta líneas con espacios dobles
structured_logs = raw_logs.select(
    F.regexp_extract("value", LOG_PATTERN, GROUP_IP).alias("ip"),
    F.regexp_extract("value", LOG_PATTERN, GROUP_TIMESTAMP).alias("timestamp"),
    F.regexp_extract("value", LOG_PATTERN, GROUP_METHOD).alias("method"),
    F.regexp_extract("value", LOG_PATTERN, GROUP_ENDPOINT).alias("endpoint"),
    F.regexp_extract("value", LOG_PATTERN, GROUP_PROTOCOL).alias("protocol"),
    F.regexp_extract("value", LOG_PATTERN, GROUP_STATUS).cast("int").alias("status_code"),
    # "-" (sin tamaño reportado) castea a NULL, no a basura.
    F.regexp_extract("value", LOG_PATTERN, GROUP_SIZE).cast("int").alias("content_size"),
)

# Una línea válida tiene IP y código de estado.
structured_logs = structured_logs.filter(
    (F.col("ip") != "") & F.col("status_code").isNotNull()
)

total = raw_logs.count()
parsed = structured_logs.count()
print(f"Registros parseados: {parsed:,}")
print(f"Líneas descartadas : {total - parsed:,} ({(total - parsed) / total * 100:.3f}%)")
structured_logs.show(5, truncate=False)

In [ ]:
# 1. SEGURIDAD: errores 4xx/5xx por IP, con tasa de fallo real.
total_por_ip = structured_logs.groupBy("ip").count().withColumnRenamed(
    "count", "total_peticiones"
)
errores_por_ip = (
    structured_logs.filter(F.col("status_code") >= 400)
    .groupBy("ip")
    .count()
    .withColumnRenamed("count", "errores")
)
reporte_seguridad = (
    total_por_ip.join(errores_por_ip, on="ip", how="inner")
    .withColumn(
        "tasa_error_pct",
        F.round(F.col("errores") / F.col("total_peticiones") * 100, 2),
    )
    .orderBy(F.col("errores").desc(), F.col("tasa_error_pct").desc())
)
print("🚨 TOP 5 IPs CON MÁS ERRORES (posibles atacantes/bots):")
reporte_seguridad.show(5, truncate=False)

In [ ]:
# 2. CONTENIDO: recursos más solicitados.
reporte_paginas = (
    structured_logs.groupBy("endpoint")
    .agg(
        F.count("*").alias("peticiones"),
        F.countDistinct("ip").alias("ips_distintas"),
    )
    .orderBy(F.col("peticiones").desc())
)
print("📑 TOP 5 RECURSOS MÁS SOLICITADOS:")
reporte_paginas.show(5, truncate=False)

In [ ]:
# 3. TRÁFICO TEMPORAL: peticiones por hora del día.
# La hora son los caracteres 13-14 de "01/Aug/1995:00:00:01 -0400".
reporte_trafico = (
    structured_logs.withColumn("hour", F.substring("timestamp", 13, 2))
    .groupBy("hour")
    .agg(
        F.count("*").alias("peticiones"),
        F.countDistinct("ip").alias("ips_distintas"),
    )
    .orderBy("hour")
)
print("⏰ TRÁFICO POR HORA DEL DÍA:")
reporte_trafico.show(24, truncate=False)

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

df_plot = reporte_trafico.toPandas()
ax = df_plot.plot(
    kind="line", x="hour", y="peticiones", marker="o", color="#1f77b4", figsize=(10, 4)
)
ax.set_title("Peticiones por hora del día - servidores HTTP NASA (Ago-1995)")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Número de peticiones")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from nasa_logs import OUTPUT_FILES
from nasa_logs_pipeline import OUTPUT_DIR, write_single_csv

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Escritura escalable: NO se vuelca el dataset completo al driver con toPandas().
write_single_csv(structured_logs, OUTPUT_DIR / OUTPUT_FILES["estructurado"])
reporte_seguridad.limit(100).toPandas().to_csv(OUTPUT_DIR / OUTPUT_FILES["seguridad"], index=False)
reporte_trafico.toPandas().to_csv(OUTPUT_DIR / OUTPUT_FILES["trafico"], index=False)
reporte_paginas.limit(100).toPandas().to_csv(OUTPUT_DIR / OUTPUT_FILES["paginas"], index=False)

print(f"✅ Ficheros generados en {OUTPUT_DIR}:")
for name in OUTPUT_FILES.values():
    f = OUTPUT_DIR / name
    print(f"  - {name} ({f.stat().st_size / 1e6:.1f} MB)")

In [ ]:
spark.stop()
print("Sesión de Spark detenida")